# Mindcore Interactive Demo

This notebook demonstrates the core features of Mindcore v2:
- Memory storage and recall
- FLR (Fast Learning Recall) caching
- CLST (Cognitive Long-term Storage Transfer)
- SVL (Shared Vocabulary Layer)
- Multi-agent memory sharing
- REST API and MCP integration

In [ ]:
# Setup
import sys


sys.path.insert(0, "../..")

import json

from mindcore.v2 import Mindcore
from mindcore.v2.svl import SharedVocabularyLayer

## 1. Basic Usage

Create a Mindcore instance and store your first memory.

In [ ]:
# Create Mindcore instance with SQLite
mc = Mindcore(storage="sqlite:///demo.db")

print("Mindcore initialized!")
print(f"Stats: {mc.get_stats()}")

In [ ]:
# Store a memory
memory_id = mc.store(
    content="User prefers Python over JavaScript for backend development",
    memory_type="preference",
    user_id="demo_user",
    topics=["api", "integration"],
    categories=["technical"],
    importance=0.9,
    entities=["Python", "JavaScript"],
)

print(f"Stored memory with ID: {memory_id}")

In [ ]:
# Recall the memory
result = mc.recall(query="What programming language does the user prefer?", user_id="demo_user")

print(f"Found {len(result.memories)} memories:")
for i, memory in enumerate(result.memories):
    print(f"  {i+1}. [{memory.memory_type}] {memory.content}")
    print(f"     Score: {result.scores[i]:.2f}, Importance: {memory.importance}")

## 2. Memory Types

Mindcore supports different memory types for different kinds of information.

In [ ]:
# Store different memory types
memories = [
    {
        "content": "User called support on Monday about login issues",
        "memory_type": "episodic",
        "topics": ["issue", "help"],
    },
    {
        "content": "API rate limit is 1000 requests per hour",
        "memory_type": "semantic",
        "topics": ["api", "documentation"],
    },
    {
        "content": "To reset password: 1. Click forgot 2. Enter email 3. Check inbox",
        "memory_type": "procedural",
        "topics": ["help", "account"],
    },
    {
        "content": "John Smith is the account manager at john@company.com",
        "memory_type": "entity",
        "topics": ["account"],
        "entities": ["John Smith", "john@company.com"],
    },
]

for mem in memories:
    mid = mc.store(
        content=mem["content"],
        memory_type=mem["memory_type"],
        user_id="demo_user",
        topics=mem["topics"],
        entities=mem.get("entities", []),
    )
    print(f"Stored {mem['memory_type']}: {mid[:20]}...")

## 3. Reinforcement Learning

Reinforce memories based on feedback to improve recall ranking.

In [ ]:
# Get initial reinforcement score
memory = mc.get(memory_id)
print(f"Initial reinforcement score: {memory.reinforcement_score}")

# Apply positive reinforcement (memory was helpful)
mc.reinforce(memory_id, 0.8)

# Check updated score
memory = mc.get(memory_id)
print(f"After positive feedback: {memory.reinforcement_score}")

# Apply negative reinforcement (memory was not useful)
mc.reinforce(memory_id, -0.3)

memory = mc.get(memory_id)
print(f"After negative feedback: {memory.reinforcement_score}")

## 4. Search with Filters

Search memories with various filters.

In [ ]:
# Search by topics
results = mc.search(user_id="demo_user", topics=["api"], limit=5)

print(f"Found {len(results)} memories with 'api' topic:")
for mem in results:
    print(f"  - {mem.content[:50]}...")

In [ ]:
# Search by memory type
results = mc.search(user_id="demo_user", memory_types=["preference"], limit=5)

print(f"Found {len(results)} preference memories:")
for mem in results:
    print(f"  - {mem.content}")

## 5. SVL (Shared Vocabulary Layer)

Get the JSON schema for structured LLM output.

In [ ]:
# Get JSON schema for LLM integration
schema = mc.get_json_schema()

print("JSON Schema for LLM structured output:")
print(json.dumps(schema, indent=2)[:500] + "...")

In [ ]:
# Get vocabulary instructions for prompts
instructions = mc.get_vocabulary_instructions()

print("Vocabulary Instructions for LLM:")
print(instructions[:500] + "...")

## 6. Multi-Agent Mode

Enable multi-agent support for team-based memory sharing.

In [ ]:
# Create multi-agent Mindcore
mc_multi = Mindcore(storage="sqlite:///demo_multi.db", enable_multi_agent=True)

# Register agents
mc_multi.register_agent(
    agent_id="support_bot",
    name="Support Agent",
    description="Handles customer support",
    teams=["support", "general"],
)

mc_multi.register_agent(
    agent_id="tech_bot",
    name="Technical Agent",
    description="Handles technical issues",
    teams=["support", "engineering"],
)

print("Registered agents:")
for agent in mc_multi.list_agents():
    print(f"  - {agent.name} (teams: {agent.teams})")

In [ ]:
# Store team-shared memory
memory_id = mc_multi.store(
    content="User reported critical bug in checkout flow",
    memory_type="episodic",
    user_id="customer_123",
    topics=["bug", "issue"],
    access_level="team",
    agent_id="support_bot",
)

print(f"Stored team memory: {memory_id}")

# Tech bot can recall it (same team)
result = mc_multi.recall(query="checkout bug", user_id="customer_123", agent_id="tech_bot")

print(f"\nTech bot found {len(result.memories)} memories:")
for mem in result.memories:
    print(f"  - {mem.content}")

## 7. Compression

Compress old memories to save space.

In [ ]:
# Store many test memories
for i in range(20):
    mc.store(
        content=f"Test memory {i} for compression demo",
        memory_type="semantic",
        user_id="compress_user",
        topics=["api"],
    )

print("Stored 20 test memories")

# Compress using deduplication
result = mc.compress(
    user_id="compress_user",
    older_than_days=0,  # All memories
    strategy="deduplicate",
)

print("\nCompression result:")
print(f"  Original count: {result.original_count}")
print(f"  Compressed count: {result.compressed_count}")
print(f"  Compression ratio: {result.compression_ratio:.2f}")

## 8. Custom Vocabulary Domain

Create a custom vocabulary for your domain.

In [ ]:
# Create custom SVL

custom_vocab = SharedVocabularyLayer(
    domains=["ecommerce"]  # Load e-commerce domain
)

# Add custom topics
custom_vocab.add_topics("loyalty_program", "flash_sale", "wishlist")
custom_vocab.add_categories("vip_support")

print("Custom vocabulary configured:")
print(f"  Active domains: {custom_vocab.get_active_domains()}")

# Generate TypeScript types
ts_types = custom_vocab.to_typescript()
print("\nGenerated TypeScript types:")
print(ts_types[:300] + "...")

## 9. Cleanup

In [ ]:
# Close connections
mc.close()
mc_multi.close()

# Optionally delete demo databases
# os.remove("demo.db")
# os.remove("demo_multi.db")

print("Demo complete!")

## Summary

This demo covered:

1. **Basic Usage**: Store and recall memories with Mindcore
2. **Memory Types**: Episodic, semantic, procedural, preference, entity
3. **Reinforcement**: Improve recall with feedback signals
4. **Search**: Filter by topics, categories, memory types
5. **SVL**: JSON schema and vocabulary for LLM integration
6. **Multi-Agent**: Team-based memory sharing and isolation
7. **Compression**: Reduce storage with deduplication/summarization
8. **Custom Domains**: Extend vocabulary for your use case

For more information, see the full documentation and test suite.